In [1]:
!pip install -q sentence-transformers shap kaggle

In [2]:
import os, re, string, random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

import tensorflow as tf
from tensorflow.keras import layers, models

from sentence_transformers import SentenceTransformer

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [6]:
df = pd.read_json("/content/resumes_dataset.jsonl", lines=True)
print(df.shape)
print(df.columns)
df.head()

df = df.dropna(subset=["Text", "Category"]).reset_index(drop=True)
df = df.rename(columns={"Text": "Resume"})

(3500, 12)
Index(['ResumeID', 'Category', 'Name', 'Email', 'Phone', 'Location', 'Summary',
       'Skills', 'Experience', 'Education', 'Text', 'Source'],
      dtype='object')


In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_resume"] = df["Resume"].apply(clean_text)
df = df[df["clean_resume"].str.len() > 20].reset_index(drop=True)

le_categories = sorted(df["Category"].unique())
cat2id = {c: i for i, c in enumerate(le_categories)}
id2cat = {i: c for c, i in cat2id.items()}
df["label"] = df["Category"].map(cat2id)

print("num categories:", len(le_categories))

num categories: 36


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_resume"], df["label"], test_size=0.2,
    stratify=df["label"], random_state=42
)

In [9]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [10]:
ml_model = LogisticRegression(max_iter=1000, C=2.0)
ml_model.fit(X_train_tfidf, y_train)

LogisticRegression(C=2.0, max_iter=1000)

In [11]:
y_pred = ml_model.predict(X_test_tfidf)
print("ML model accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le_categories, zero_division=0))

ML model accuracy: 0.89
                           precision    recall  f1-score   support

              AI Engineer       1.00      1.00      1.00        14
        Backend Developer       1.00      1.00      1.00        15
               Blockchain       1.00      0.70      0.82        10
     Blockchain Developer       1.00      1.00      1.00         6
         Business Analyst       0.86      1.00      0.92        30
           Cloud Engineer       1.00      1.00      1.00        19
    Cybersecurity Analyst       1.00      1.00      1.00        13
             Data Science       0.84      0.95      0.89        40
                 Database       0.85      0.57      0.68        30
   Database Administrator       1.00      1.00      1.00        11
                   DevOps       0.97      1.00      0.99        36
            Digital Media       0.88      0.70      0.78        20
         DotNet Developer       0.79      0.82      0.81        28
            ETL Developer       0.75 

In [12]:
n_classes = len(le_categories)

dl_model = models.Sequential([
    layers.Input(shape=(X_train_tfidf.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(n_classes, activation="softmax")
])

dl_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [13]:
history = dl_model.fit(
    X_train_tfidf.toarray(), y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=32,
    verbose=1
)

dl_loss, dl_acc = dl_model.evaluate(X_test_tfidf.toarray(), y_test, verbose=0)
print("DL model accuracy:", dl_acc)

Epoch 1/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.3063 - loss: 2.8216 - val_accuracy: 0.6929 - val_loss: 1.7004
Epoch 2/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.7548 - loss: 1.1456 - val_accuracy: 0.8286 - val_loss: 0.7495
Epoch 3/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8754 - loss: 0.5479 - val_accuracy: 0.8643 - val_loss: 0.5521
Epoch 4/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9290 - loss: 0.3132 - val_accuracy: 0.8714 - val_loss: 0.4850
Epoch 5/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9611 - loss: 0.1949 - val_accuracy: 0.8714 - val_loss: 0.4537
Epoch 6/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9611 - loss: 0.1569 - val_accuracy: 0.8786 - val_loss: 0.4572
Epoch 7/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9702 - loss: 0.1115 - val_accuracy: 0.8750 - val_loss: 0.4879
Epoch 8/8
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9810 - loss: 0.0922 - val_accuracy: 0.8679 - val_loss:

In [14]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

df["embedding"] = list(embedder.encode(df["clean_resume"].tolist(), show_progress_bar=True))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

In [15]:
def get_match_score(resume_text, jd_text):
    resume_emb = embedder.encode([clean_text(resume_text)])
    jd_emb = embedder.encode([clean_text(jd_text)])
    score = cosine_similarity(resume_emb, jd_emb)[0][0]
    return round(float(score) * 100, 2)

In [16]:
sample_jd = "Looking for a data scientist with experience in python, machine learning, sql and deep learning."
sample_resume = df.iloc[0]["Resume"]
print("match score example:", get_match_score(sample_resume, sample_jd))

match score example: 20.94


In [17]:
feature_names = np.array(tfidf.get_feature_names_out())

def explain_ml_prediction(resume_text, top_n=8):
    clean = clean_text(resume_text)
    vec = tfidf.transform([clean])
    pred_id = ml_model.predict(vec)[0]
    pred_label = id2cat[pred_id]

    coefs = ml_model.coef_[pred_id]
    nonzero_idx = vec.nonzero()[1]
    contribs = [(feature_names[i], coefs[i] * vec[0, i]) for i in nonzero_idx]
    contribs = sorted(contribs, key=lambda x: x[1], reverse=True)[:top_n]

    return pred_label, contribs

pred_label, contribs = explain_ml_prediction(sample_resume)
print("predicted category:", pred_label)
print("top contributing terms:", contribs)

predicted category: Java Developer
top contributing terms: [('java', np.float64(1.4359285280277925)), ('java developer', np.float64(0.35451418306736965)), ('john', np.float64(0.16361229615329062)), ('senior java', np.float64(0.14664215141590448)), ('web', np.float64(0.09319269687960412)), ('jdbc', np.float64(0.06497227206876642)), ('software', np.float64(0.06263411239227215)), ('application', np.float64(0.05864962655009232))]


In [18]:
feedback_bank = [
    "Your resume lacks measurable achievements. Try adding numbers, percentages or outcomes to your bullet points.",
    "Consider adding more technical keywords related to the job description to pass ATS screening.",
    "Your experience section is strong but could use clearer action verbs at the start of each bullet.",
    "Add a dedicated skills section listing tools and technologies relevant to the role you're targeting.",
    "Your resume seems to lack certifications. Adding relevant certifications can strengthen your profile.",
    "The resume format looks dense. Break large paragraphs into concise bullet points for readability.",
    "Consider highlighting leadership or team collaboration experience if applying for senior roles.",
    "Include project links or portfolio references to back up your technical claims.",
    "Your resume matches the job description well on core skills, keep it up.",
    "Try tailoring your summary section to directly reflect the job title and top 3 requirements.",
    "Missing keywords around cloud platforms (AWS, GCP, Azure) if the job requires them.",
    "Quantify your impact - e.g. 'reduced processing time by 30%' instead of just 'improved processing'."
]

In [19]:
feedback_vectorizer = TfidfVectorizer(stop_words="english")
feedback_matrix = feedback_vectorizer.fit_transform(feedback_bank)

def generate_feedback(resume_text, jd_text, top_k=3):
    match_score = get_match_score(resume_text, jd_text)
    pred_label, contribs = explain_ml_prediction(resume_text)

    query = clean_text(resume_text) + " " + clean_text(jd_text)
    query_vec = feedback_vectorizer.transform([query])
    sims = cosine_similarity(query_vec, feedback_matrix)[0]
    top_idx = sims.argsort()[::-1][:top_k]

    retrieved = [feedback_bank[i] for i in top_idx]

    report = []
    report.append(f"Predicted resume category: {pred_label}")
    report.append(f"Match score with job description: {match_score}%")
    report.append(f"Key terms influencing this match: {', '.join([c[0] for c in contribs[:5]])}")
    report.append("Personalized suggestions:")
    for r in retrieved:
        report.append(f"- {r}")

    return "\n".join(report)

print(generate_feedback(sample_resume, sample_jd))

Predicted resume category: Java Developer
Match score with job description: 20.94%
Key terms influencing this match: java, java developer, john, senior java, web
Personalized suggestions:
- Consider highlighting leadership or team collaboration experience if applying for senior roles.
- Try tailoring your summary section to directly reflect the job title and top 3 requirements.
- Include project links or portfolio references to back up your technical claims.


In [20]:
class ResumeAssistant:
    def __init__(self, resume_text, jd_text):
        self.resume_text = resume_text
        self.jd_text = jd_text
        self.match_score = get_match_score(resume_text, jd_text)
        self.pred_label, self.contribs = explain_ml_prediction(resume_text)
        self.feedback = generate_feedback(resume_text, jd_text)

    def respond(self, query):
        q = query.lower()

        if "score" in q or "match" in q:
            return f"Your resume matches the job description with a score of {self.match_score}%."
        elif "category" in q or "domain" in q or "field" in q:
            return f"Your resume seems to fall under the '{self.pred_label}' category."
        elif "feedback" in q or "improve" in q or "suggestion" in q:
            return self.feedback
        elif "keyword" in q or "term" in q:
            terms = ", ".join([c[0] for c in self.contribs[:6]])
            return f"The strongest terms in your resume are: {terms}."
        elif "hello" in q or "hi" in q:
            return "Hi! I'm your resume assistant. Ask me about your match score, category, or feedback."
        else:
            return "I can help with your resume's match score, predicted category, or improvement feedback. What would you like to know?"

In [21]:
assistant = ResumeAssistant(sample_resume, sample_jd)

demo_queries = [
    "hi",
    "what's my match score?",
    "what category does my resume fall under?",
    "give me feedback",
    "what are my top keywords?"
]

for q in demo_queries:
    print("User:", q)
    print("Bot:", assistant.respond(q))
    print()

User: hi
Bot: Hi! I'm your resume assistant. Ask me about your match score, category, or feedback.

User: what's my match score?
Bot: Your resume matches the job description with a score of 20.94%.

User: what category does my resume fall under?
Bot: Your resume seems to fall under the 'Java Developer' category.

User: give me feedback
Bot: Predicted resume category: Java Developer
Match score with job description: 20.94%
Key terms influencing this match: java, java developer, john, senior java, web
Personalized suggestions:
- Consider highlighting leadership or team collaboration experience if applying for senior roles.
- Try tailoring your summary section to directly reflect the job title and top 3 requirements.
- Include project links or portfolio references to back up your technical claims.

User: what are my top keywords?
Bot: The strongest terms in your resume are: java, java developer, john, senior java, web, jdbc.

